In [9]:
import pandas as pd
import numpy as np
import csv
from scipy import stats
import os


def corr(input_file, output_file):
    
    # Read the file (no header)
    df = pd.read_csv(input_file, header=None, na_values=["", "NaN"], keep_default_na=True)

    # Keep first row
    first_row = df.iloc[[0]]
    
    # Separate CPX and META rows but keep names
    cpx_rows = df[df[0].astype(str).str.endswith('-P')]
    meta_rows = df[~df[0].astype(str).str.endswith('-P')].iloc[1:]
    #print(meta_rows)
    
    # Print the first CPX row including its name
    #print("First CPX row:")
    #print(cpx_rows.iloc[[0]])
    
    # Extract metabolite names
    cpx_list = cpx_rows.iloc[:, 0].tolist()
    meta_list = meta_rows.iloc[:, 0].tolist()
    #print(meta_list)
    #print(cpx_list)
    
    # Now drop name column to get numeric values only
    cpx = cpx_rows.iloc[:, 1:]
    meta = meta_rows.iloc[:, 1:]


    results = []
    temp = 0
    
    # Build CPX × META pairs
    pairs = []
    for m1 in cpx_list:
        for m2 in meta_list:
            pairs.append((m1, m2))
    #for pair in pairs[:5]:
        #print(pair)


    #for met1, met2 in pairs:
    for met1, met2 in [("ENSMUSG00000000127-P", "1-(2,4-Dimethoxyphenyl)-propan-2-one")]:
        # Extract rows with values (col 2+)
        l1 = df[df.iloc[:,0] == met1].iloc[:,1:]
        l2 = df[df.iloc[:,0] == met2].iloc[:,1:]
    
        # Convert to float arrays
        row1 = l1.astype(float).values.flatten()
        row2 = l2.astype(float).values.flatten()
        
        #print(row1)
        #print(row2)
    
        # Mask positions where BOTH values exist
        mask = ~np.isnan(row1) & ~np.isnan(row2)
        r1_masked = row1[mask]
        r2_masked = row2[mask]
        print(stats.rankdata(r1_masked),r1_masked)
        print(stats.rankdata(r2_masked),r2_masked)
    
        # Too few points → NaN
        if len(r1_masked) < 3:
            corr, pval = np.nan, np.nan
            count = 0
        else:
            corr, pval = stats.spearmanr(r1_masked, r2_masked)
            count = mask.sum()
    
        temp += 1
        results.append((met1, met2, corr, pval, count))


    
    # Create output folder
    os.makedirs("5. Metabolite Pairs", exist_ok=True)  
    output_file_path = os.path.join("5. Metabolite Pairs", output_file) 
    # Write correlation results
    with open(output_file_path, "w", newline='', encoding="utf-8") as outfile:
        writer = csv.writer(outfile)
        writer.writerow(["Metabolite 1", "Metabolite 2", "Spearman Coefficient", "p-value", "Values Counted"])
        writer.writerows(results)

#corr("./4. Combine Data Tables/dss_combined.csv","dss_met_pair.csv")
#print("done")
#corr("./4. Combine Data Tables/lps_combined.csv","lps_met_pair.csv")
#print("done")
#corr("./4. Combine Data Tables/vecpac_combined.csv","vecpac_met_pair.csv")
#print("done")
corr("./4. Combine Data Tables/all_combined.csv","all_met_pair.csv")


[ 3. 13.  2.  8. 16. 10. 12.  7. 17. 15.  5. 14.  9. 11.  4.  1.  6.] [4.79976788 5.69296584 4.59851841 5.11572059 6.04734841 5.30744954
 5.49694706 5.08793353 6.11360164 5.99766914 4.90166945 5.77620676
 5.22856164 5.45555774 4.86265045 4.57050504 4.99319838]
[ 1.  17.   7.   2.   4.5  4.5  4.5  4.5 11.  12.   8.5 13.   8.5 14.5
 10.  14.5 16. ] [22.14273214 23.00170775 22.27119795 22.19344393 22.22411493 22.22411493
 22.22411493 22.22411493 22.47529504 22.65931447 22.42586394 22.71443284
 22.42586394 22.79378707 22.4618524  22.79378707 22.98560859]
